In [ ]:
# Import erforderlicher Bibliotheken
import pandas as pd
import numpy as np
import chardet
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns
import chardet
import sys

# Setze Seed für Reproduzierbarkeit
np.random.seed(42)

# Funktion zur Erkennung der Dateicodierung
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    return result['encoding']

# Funktion zum Einlesen der CSV mit Fehlerbehandlung
def load_csv(file_path):
    try:
        # Zuerst mit erkanntem Encoding versuchen
        encoding = detect_encoding(file_path)
        print(f"Versuch mit erkanntem Encoding: {encoding}")
        df = pd.read_csv(file_path, encoding=encoding, on_bad_lines='warn', sep=None, engine='python')
        return df
    except Exception as e:
        print(f"Fehler mit erkanntem Encoding: {e}")
        print("Versuche mit alternativen Encodings...")
        
        encodings = ['utf-8', 'latin1', 'ISO-8859-1', 'cp1252', 'utf-16']
        
        for enc in encodings:
            try:
                print(f"Versuche mit: {enc}")
                df = pd.read_csv(file_path, encoding=enc, on_bad_lines='warn', sep=None, engine='python')
                print(f"Erfolgreich mit: {enc}")
                return df
            except Exception as e:
                print(f"Fehler mit {enc}: {e}")
                continue
        
        # Letzter Versuch mit Semikolon als Trennzeichen
        try:
            print("Versuche mit Semikolon als Trennzeichen...")
            df = pd.read_csv(file_path, sep=';', encoding='latin1', on_bad_lines='warn')
            return df
        except Exception as e:
            print(f"Fehler mit Semikolon als Trennzeichen: {e}")
        
        raise ValueError("Konnte die Datei nicht mit den verfügbaren Encodings einlesen")

# 1. Daten einlesen
try:
    file_path = '/Users/dennyredel/Documents/HFT/Data Analytics/project/survey-test.CSV'
    print(f"Lese Datei: {file_path}")
    df = load_csv(file_path)
    
    print("\nErfolgreich eingelesen! Erste Zeilen des Datensatzes:")
    display(df.head(3))
    print("\nSpalten im Datensatz:", df.columns.tolist())
    
except Exception as e:
    print(f"Kritischer Fehler: {str(e)}")
    print("""
    Bitte überprüfe folgende Punkte:
    1. Existiert die Datei unter dem angegebenen Pfad?
    2. Ist die Datei möglicherweise beschädigt?
    3. Kannst du die Datei in einem Texteditor öffnen und den Inhalt überprüfen?
    """)
    sys.exit(1)

# 2. Datenvorbereitung
print("\nBereite die Daten vor...")

# Features und Zielvariable
features = [
    'Age', 'EdLevel', 'Employment', 'WorkExp', 'YearsCode',
    'DevType', 'OrgSize', 'RemoteWork', 'Country', 'CompTotal'
]
target = 'JobSat'

# Überprüfe, ob alle benötigten Spalten vorhanden sind
missing_columns = [col for col in features + [target] if col not in df.columns]
if missing_columns:
    print(f"Warnung: Folgende Spalten fehlen im Datensatz: {missing_columns}")
    print("Verfügbare Spalten:", df.columns.tolist())
    sys.exit(1)

X = df[features].copy()
y = df[target].copy()

# Datenbereinigung
print("\nFühre Datenbereinigung durch...")

# Konvertiere numerische Spalten
numeric_columns = ['YearsCode', 'WorkExp', 'Age', 'CompTotal']
for col in numeric_columns:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')

# Fehlende Werte behandeln
y = y.dropna()
X = X.loc[y.index]

# Numerische Spalten mit Median auffüllen
for col in X.select_dtypes(include=['number']).columns:
    X[col] = X[col].fillna(X[col].median())

# Kategorische Spalten mit 'Missing' auffüllen
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].fillna('Missing')

# Klassenverteilung anzeigen
print("\nKlassenverteilung der Zielvariable:")
print(y.value_counts(normalize=True))

# 3. Train/Test-Split
print("\nFühre Train/Test-Split durch...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 4. Vorverarbeitungspipeline
print("\nErstelle Vorverarbeitungspipeline...")

# Numerische und kategorische Features identifizieren
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numerische Features: {numerical_features}")
print(f"Kategorische Features: {categorical_features}")

# Preprocessing-Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

# 5. Modellpipeline erstellen
print("\nErstelle Modellpipeline...")
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(probability=True, random_state=42))
])

# 6. Hyperparameter-Tuning mit GridSearchCV
print("\nStarte GridSearchCV für Hyperparameter-Tuning...")
param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__gamma': ['scale', 'auto'],
    'classifier__kernel': ['rbf']
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring='f1_weighted'
)

# Modelltraining
print("\nStarte Modelltraining...")
grid_search.fit(X_train, y_train)

# 7. Ergebnisse auswerten
print("\nBeste Parameter gefunden:")
print(grid_search.best_params_)
print(f"Beste Cross-Validation Score: {grid_search.best_score_:.3f}")

# Vorhersagen auf dem Testset
print("\nErgebnisse auf dem Testset:")
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))

# Konfusionsmatrix
cm = confusion_matrix(y_test, y_pred)
print("Konfusionsmatrix:")
print(cm)

# Visualisierung der Konfusionsmatrix
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Konfusionsmatrix')
plt.ylabel('Wahre Klasse')
plt.xlabel('Vorhergesagte Klasse')
plt.show()

# Cross-Validation auf dem Trainingsset
print("\nErgebnisse der Cross-Validation auf dem Trainingsset:")
y_cv = cross_val_predict(grid_search.best_estimator_, X_train, y_train, cv=5)
print(classification_report(y_train, y_cv))

# Feature Importance (nur für linearen Kernel)
if grid_search.best_params_['classifier__kernel'] == 'linear':
    print("\nBerechne Feature-Importance...")
    try:
        # Feature-Namen nach One-Hot-Encoding
        ohe_columns = (grid_search.best_estimator_.named_steps['preprocessor']
                      .named_transformers_['cat']
                      .get_feature_names_out(categorical_features))
        
        all_features = np.concatenate([numerical_features, ohe_columns])
        
        # Koeffizienten extrahieren
        coef = grid_search.best_estimator_.named_steps['classifier'].coef_
        
        # DataFrame für Feature-Importance erstellen
        feature_importance = pd.DataFrame({
            'Feature': all_features,
            'Wichtigkeit': np.abs(coef).mean(axis=0)
        }).sort_values('Wichtigkeit', ascending=False)
        
        # Top 20 wichtigste Features visualisieren
        plt.figure(figsize=(12, 8))
        sns.barplot(x='Wichtigkeit', y='Feature', 
                    data=feature_importance.head(20))
        plt.title('Top 20 Wichtigste Features')
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Konnte Feature-Importance nicht berechnen: {e}")

# 8. Modell speichern
print("\nSpeichere das trainierte Modell...")
import joblib
joblib.dump(grid_search.best_estimator_, 'svm_model.pkl')
print("Modell wurde erfolgreich als 'svm_model.pkl' gespeichert.")

print("\nModelltraining und -auswertung abgeschlossen!")

Lese Datei: /Users/dennyredel/Documents/HFT/Data Analytics/project/survey-test.CSV
Versuch mit erkanntem Encoding: Windows-1252

Erfolgreich eingelesen! Erste Zeilen des Datensatzes:


,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",None of the above,10,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,"Understand how things actually work, problem s...",53061.0,8.0



Spalten im Datensatz: ['ResponseId', 'MainBranch', 'Age', 'EdLevel', 'Employment', 'EmploymentAddl', 'WorkExp', 'LearnCodeChoose', 'LearnCode', 'LearnCodeAI', 'AILearnHow', 'YearsCode', 'DevType', 'OrgSize', 'ICorPM', 'RemoteWork', 'PurchaseInfluence', 'TechEndorseIntro', 'TechEndorse_1', 'TechEndorse_2', 'TechEndorse_3', 'TechEndorse_4', 'TechEndorse_5', 'TechEndorse_6', 'TechEndorse_7', 'TechEndorse_8', 'TechEndorse_9', 'TechEndorse_13', 'TechEndorse_13_TEXT', 'TechOppose_1', 'TechOppose_2', 'TechOppose_3', 'TechOppose_5', 'TechOppose_7', 'TechOppose_9', 'TechOppose_11', 'TechOppose_13', 'TechOppose_16', 'TechOppose_15', 'TechOppose_15_TEXT', 'Industry', 'JobSatPoints_1', 'JobSatPoints_4', 'JobSatPoints_5', 'JobSatPoints_6', 'JobSatPoints_7', 'JobSatPoints_8', 'JobSatPoints_9', 'JobSatPoints_10', 'JobSatPoints_11', 'JobSatPoints_13', 'JobSatPoints_14', 'JobSatPoints_15', 'JobSatPoints_16', 'JobSatPoints_15_TEXT', 'AIThreat', 'NewRole', 'ToolCountWork', 'ToolCountPersonal', 'Country'

/Users/dennyredel/Documents/HFT/Data Analytics/venv/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.